# Experiment: daily_stock Top500 Coverage and Forward Availability Review

Objective:
- Review the separate forward-coverage artifact from `profile_daily_stock_forward_coverage.py`.
- Answer the whole-timeline rolling top-500 coverage question.
- Diagnose evaluator-style next-day return availability in the current 2018-2020 sample window.

Interpretation rule: this is data-understanding evidence, not alpha evidence and not a child promotion decision.


In [ ]:
from __future__ import annotations

import json
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd())
ARTIFACT_CANDIDATES = [
    REPO_ROOT / "artifacts" / "phase4_alphaevolve" / "daily_stock_forward_coverage_20260518",
    REPO_ROOT / "artifacts" / "daily_stock_forward_coverage_20260518",
    REPO_ROOT / "artifacts" / "phase4_alphaevolve" / "daily_stock_forward_coverage_20260518.zip",
    REPO_ROOT / "artifacts" / "daily_stock_forward_coverage_20260518.zip",
]

def resolve_artifact(candidates: list[Path]) -> Path:
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
        if candidate.is_file() and candidate.suffix.lower() == ".zip":
            extract_root = REPO_ROOT / ".tmp" / "notebooks" / candidate.stem
            extract_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(candidate) as zf:
                zf.extractall(extract_root)
            nested = [path for path in extract_root.rglob("forward_coverage_summary.json")]
            if nested:
                return nested[0].parent
            return extract_root
    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find the forward-coverage artifact. Searched:\n{searched}")

ARTIFACT = resolve_artifact(ARTIFACT_CANDIDATES)
ARTIFACT


## Artifact Contract

Expected files:
- `forward_coverage_summary.json`
- `top500_daily_coverage.csv`
- `top500_monthly_coverage.csv`
- `top500_membership_monthly.csv`
- `top500_membership_churn.csv`
- `top500_permno_coverage.csv`
- `forward_availability_by_date.csv`
- `forward_availability_by_bucket.csv`
- `forward_availability_by_industry.csv`
- `forward_availability_by_exchange.csv`
- `held_availability_prompt_cards.md`


In [ ]:
def read_json(name: str) -> dict:
    return json.loads((ARTIFACT / name).read_text(encoding="utf-8"))

def read_csv(name: str, **kwargs) -> pd.DataFrame:
    path = ARTIFACT / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path, **kwargs)

summary = read_json("forward_coverage_summary.json")
daily = read_csv("top500_daily_coverage.csv", parse_dates=["date"])
monthly = read_csv("top500_monthly_coverage.csv", parse_dates=["month"])
membership = read_csv("top500_membership_monthly.csv", parse_dates=["month", "formation_date"])
churn = read_csv("top500_membership_churn.csv", parse_dates=["month"])
permno_coverage = read_csv("top500_permno_coverage.csv")
availability_date = read_csv("forward_availability_by_date.csv", parse_dates=["date"])
availability_bucket = read_csv("forward_availability_by_bucket.csv")
availability_industry = read_csv("forward_availability_by_industry.csv")
availability_exchange = read_csv("forward_availability_by_exchange.csv")

summary


## One-Screen Overview

Use this table for meeting discussion before looking at plots.


In [ ]:
forward = summary.get("forward_availability", {})
overview = pd.DataFrame(
    [
        {"metric": "top_n", "value": summary.get("top_n")},
        {"metric": "raw_trading_date_count", "value": summary.get("raw_trading_date_count")},
        {"metric": "eligible_trading_date_count", "value": summary.get("eligible_trading_date_count")},
        {"metric": "topn_month_count", "value": summary.get("topn_month_count")},
        {"metric": "topn_distinct_permnos", "value": summary.get("topn_distinct_permnos")},
        {"metric": "median_daily_coverage_rate", "value": summary.get("daily_coverage", {}).get("median_coverage_rate")},
        {"metric": "min_daily_coverage_rate", "value": summary.get("daily_coverage", {}).get("min_coverage_rate")},
        {"metric": "median_monthly_coverage_rate", "value": summary.get("monthly_coverage", {}).get("median_coverage_rate")},
        {"metric": "median_membership_jaccard", "value": summary.get("membership_churn", {}).get("median_jaccard_vs_prior")},
        {"metric": "forward_window", "value": f"{forward.get('start_date')} to {forward.get('end_date')}"},
        {"metric": "forward_availability_rate", "value": forward.get("availability_rate")},
        {"metric": "forward_unavailable_count", "value": forward.get("unavailable_count")},
    ]
)
overview


## Whole-Timeline Rolling Top-500 Coverage

The top-500 universe is selected monthly from lagged market cap. Coverage here measures whether selected names are observed on each eligible trading date through the available timeline.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
daily.plot(x="date", y=["observed_count", "selected_count"], ax=axes[0], linewidth=0.8)
axes[0].set_title("Daily observed selected names vs monthly selected count")
axes[0].set_ylabel("names")
monthly.plot(x="month", y="coverage_rate", ax=axes[1], linewidth=0.9, legend=False)
axes[1].set_title("Monthly top-500 row coverage rate")
axes[1].set_ylabel("coverage rate")
axes[1].set_ylim(max(0, monthly["coverage_rate"].min() - 0.02), 1.01)
plt.tight_layout()


In [ ]:
worst_days = daily.sort_values(["coverage_rate", "observed_count"]).head(20)
worst_months = monthly.sort_values(["coverage_rate", "median_daily_observed_count"]).head(20)
display(worst_days)
display(worst_months)


## Membership Stability

This separates strategy turnover from exogenous rolling-universe turnover. High membership churn can make otherwise stable strategies look operationally noisy.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
churn.dropna(subset=["jaccard_vs_prior"]).plot(x="month", y="jaccard_vs_prior", ax=ax, linewidth=0.9, legend=False)
ax.set_title("Month-to-month top-500 membership Jaccard")
ax.set_ylabel("Jaccard vs prior month")
ax.set_ylim(max(0, churn["jaccard_vs_prior"].min(skipna=True) - 0.05), 1.01)
plt.tight_layout()

display(churn.sort_values("jaccard_vs_prior").head(20))


## Forward-Return Availability

This matches evaluator semantics: rows are available only if the security's next row is the next visible market date and the forward return exists. The generated child strategies must not read `fwd_ret`, `fwd_date`, `next_market_date`, or `one_day_forward`; these are evaluator diagnostics only.


In [ ]:
cause_counts = pd.Series(forward.get("cause_counts", {}), name="count").sort_values(ascending=False)
display(cause_counts.to_frame())

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
availability_date.plot(x="date", y="availability_rate", ax=axes[0], linewidth=0.9, legend=False)
axes[0].set_title("Daily next-day return availability rate")
axes[0].set_ylabel("availability")
unavailable_cols = [
    "security_not_observed_next_market_date",
    "missing_forward_return",
    "no_next_security_row",
    "final_visible_market_date",
]
availability_date.plot(x="date", y=[col for col in unavailable_cols if col in availability_date], ax=axes[1], linewidth=0.9)
axes[1].set_title("Daily unavailable rows by cause")
axes[1].set_ylabel("rows")
plt.tight_layout()


In [ ]:
def worst_availability(frame: pd.DataFrame, label: str, n: int = 20) -> pd.DataFrame:
    cols = ["dimension", "value", "row_count", "available_count", "unavailable_count", "availability_rate"]
    return frame.sort_values(["availability_rate", "row_count"], ascending=[True, False]).head(n)[cols]

display(worst_availability(availability_bucket, "bucket"))
display(worst_availability(availability_industry, "industry"))
display(worst_availability(availability_exchange, "exchange"))


## PERMNO-Level Persistence

Use this to see whether coverage gaps are broad market-calendar effects or concentrated in names with short top-500 tenures.


In [ ]:
display(permno_coverage.sort_values(["coverage_rate", "months_in_topn"], ascending=[True, False]).head(30))

fig, ax = plt.subplots(figsize=(9, 4))
permno_coverage["months_in_topn"].hist(bins=50, ax=ax)
ax.set_title("Distribution of months in rolling top-500")
ax.set_xlabel("months in top-500")
ax.set_ylabel("PERMNO count")
plt.tight_layout()


## Meeting Notes Template

Fill this after running the notebook against the returned artifact.

- Whole-timeline top-500 coverage answer:
- Dominant forward-unavailability cause:
- Buckets or industries with concentrated risk:
- Prompt-card lessons to promote:
- Lessons to keep as caveats only:
- Whether child generation should resume:


In [ ]:
report_stub = {
    "artifact": str(ARTIFACT),
    "whole_timeline_top500_coverage": {
        "median_daily_coverage_rate": summary.get("daily_coverage", {}).get("median_coverage_rate"),
        "min_daily_coverage_rate": summary.get("daily_coverage", {}).get("min_coverage_rate"),
        "topn_month_count": summary.get("topn_month_count"),
        "topn_distinct_permnos": summary.get("topn_distinct_permnos"),
    },
    "forward_availability": {
        "availability_rate": forward.get("availability_rate"),
        "cause_counts": forward.get("cause_counts"),
    },
    "decision": "fill_after_review",
}
report_stub
